# Semantic Search Engine - Exploration and Testing

This notebook demonstrates the complete pipeline of the Semantic Search Engine, from data loading to search evaluation.

## Table of Contents
1. Setup and Configuration
2. Data Loading and Preprocessing
3. Embedding Generation
4. FAISS Index Creation
5. Search Testing
6. Evaluation Metrics
7. Visualization

In [ ]:
# Import required libraries
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Add project root to path
sys.path.insert(0, '..')

# Set plot style
%matplotlib inline
plt.style.use('seaborn-v0_8')
sns.set_palette('viridis')

## 1. Setup and Configuration

In [ ]:
# Import project modules
from utils.config import *
from utils.logger import setup_logger
from preprocessing import TextCleaner, DataLoader
from embeddings import EmbeddingGenerator
from vector_database import FAISSIndex, build_faiss_index
from search_engine import SemanticSearchEngine
from search_engine.ranking import evaluate_search_quality
from utils.visualization import SearchVisualizer

logger = setup_logger(__name__)
print(f"Using device: {DEVICE}")
print(f"Model: {BERT_MODEL_NAME}")

## 2. Data Loading and Preprocessing

In [ ]:
# Load dataset
loader = DataLoader(dataset_name='stackoverflow')
df = loader.load_stackoverflow(sample_size=1000)

print(f"Loaded {len(df)} documents")
df.head()

In [ ]:
# Preprocess text
cleaner = TextCleaner(
    lowercase=True,
    remove_punctuation=True,
    remove_stopwords=True,
    remove_special_chars=True,
    min_length=10
)

df = cleaner.clean_dataframe(df, column_name='text', output_column='cleaned_text')
print(f"After cleaning: {len(df)} documents")

# Display sample
print("\nOriginal Text:")
print(df.iloc[0]['text'][:200])
print("\nCleaned Text:")
print(df.iloc[0]['cleaned_text'][:200])

## 3. Embedding Generation

In [ ]:
# Generate embeddings
generator = EmbeddingGenerator(model_name=BERT_MODEL_NAME, batch_size=32)

texts = df['cleaned_text'].fillna('').tolist()
embeddings = generator.generate_embeddings(texts, show_progress=True)

print(f"Embeddings shape: {embeddings.shape}")
print(f"Embedding dimension: {embeddings.shape[1]}")

In [ ]:
# Analyze embedding statistics
print(f"Mean norm: {np.linalg.norm(embeddings, axis=1).mean():.4f}")
print(f"Std norm: {np.linalg.norm(embeddings, axis=1).std():.4f}")
print(f"Min value: {embeddings.min():.4f}")
print(f"Max value: {embeddings.max():.4f}")

## 4. FAISS Index Creation

In [ ]:
# Build FAISS index
faiss_index = build_faiss_index(
    embeddings=embeddings,
    index_type='IndexFlatL2',
    metric='cosine'
)

print(f"Index created with {faiss_index.index.ntotal} vectors")
print(f"Index type: {faiss_index.index_type}")
print(f"Metric: {faiss_index.metric}")

## 5. Search Testing

In [ ]:
# Test search functionality
test_queries = [
    "machine learning algorithms",
    "web development frameworks",
    "data science tools",
    "deep learning neural networks"
]

# Create document mapping
documents = []
for idx, row in df.iterrows():
    doc = {
        'id': idx,
        'text': row.get('cleaned_text', ''),
        'title': row.get('title', '')
    }
    documents.append(doc)

doc_mapping = {i: doc for i, doc in enumerate(documents)}

In [ ]:
# Perform searches
from vector_database.index_utils import format_search_results

for query in test_queries:
    print(f"\n{'='*80}")
    print(f"Query: {query}")
    print('='*80)
    
    # Generate query embedding
    query_emb = generator.generate_embeddings(query, show_progress=False)
    
    # Search
    distances, indices = faiss_index.search(query_emb, top_k=3)
    
    # Format results
    results = format_search_results(distances, indices, doc_mapping, top_k=3)
    
    for result in results:
        print(f"\nRank {result['rank']}: Score = {result['similarity_score']:.4f}")
        print(f"Text: {result['text'][:150]}...")

## 6. Evaluation Metrics

In [ ]:
# Evaluate search quality (with mock ground truth)
query = "machine learning"
query_emb = generator.generate_embeddings(query, show_progress=False)
distances, indices = faiss_index.search(query_emb, top_k=10)

# Mock relevant documents (in real scenario, use annotated data)
relevant_ids = set(indices[0][:5].tolist())  # Assume top 5 are relevant
retrieved_ids = indices[0].tolist()

# Calculate metrics
from search_engine.ranking import (
    calculate_recall_at_k,
    calculate_precision_at_k,
    calculate_mean_reciprocal_rank
)

recall_5 = calculate_recall_at_k(relevant_ids, retrieved_ids, k=5)
precision_5 = calculate_precision_at_k(relevant_ids, retrieved_ids, k=5)
mrr = calculate_mean_reciprocal_rank(relevant_ids, retrieved_ids)

print(f"Evaluation Metrics for '{query}':")
print(f"Recall@5: {recall_5:.4f}")
print(f"Precision@5: {precision_5:.4f}")
print(f"MRR: {mrr:.4f}")

## 7. Visualization

In [ ]:
# Initialize visualizer
visualizer = SearchVisualizer()

# Get similarity scores from last search
scores = distances[0][indices[0] >= 0]

# Plot similarity distribution
fig = visualizer.plot_similarity_distribution(
    scores,
    title="Distribution of Similarity Scores for Test Query"
)
plt.show()

In [ ]:
# Visualize embeddings with t-SNE (using subset for speed)
sample_indices = np.random.choice(len(embeddings), min(500, len(embeddings)), replace=False)
embeddings_sample = embeddings[sample_indices]

fig = visualizer.plot_embedding_tsne(
    embeddings_sample,
    title="t-SNE Visualization of Document Embeddings (500 samples)"
)
plt.show()

In [ ]:
# Plot search results quality
results = format_search_results(distances, indices, doc_mapping, top_k=10)
fig = visualizer.plot_search_results_quality(
    results,
    title="Search Quality Analysis"
)
plt.show()

## Summary

In this notebook, we:
1. ✅ Loaded and preprocessed text data
2. ✅ Generated BERT embeddings using Sentence-Transformers
3. ✅ Built a FAISS index for fast similarity search
4. ✅ Tested semantic search with multiple queries
5. ✅ Evaluated search quality with standard metrics
6. ✅ Visualized embeddings and search results

### Next Steps
- Experiment with different BERT models
- Try IVF index for larger datasets
- Implement hybrid search (BM25 + BERT)
- Deploy with FastAPI and Streamlit